# Project 2：錄音 → C++ 元件 → 輸出
音訊：`callback → DSP → 輸出`；分析支線：`SPSC → worker → 參數 SPSC → callback`。

先錄 3 秒，再離線逐區塊呼叫真正的 `hpp / cpp`；DSP 目前直通，固定參數只示範 I/O。
Python 負責錄音／播放，這裡不使用 `PortAudioDevice`，也不量測即時延遲。

需要 C++20 編譯器及 `numpy sounddevice ipython`（缺套件時執行 `%pip install numpy sounddevice ipython`）。

In [1]:
# 1. 載入接線（可從 notebook 目錄或 repository 內啟動）
from pathlib import Path
import sys
folder = next(p for root in [Path.cwd(), *Path.cwd().parents]
              for p in (root, root / "notebook", root / "components/project2/notebook", root / "components/project2/notebook/src")
              if (p / "project2_demo.py").is_file())
sys.path.insert(0, str(folder))
from project2_demo import record, run_project2, show_stages
from IPython.display import Audio, display

In [2]:
# 2. 錄音：執行後對麥克風說話
recording, fs = record(seconds=3)
print(f"{recording.size:,} samples，{fs} Hz，mono float32")
Audio(recording, rate=fs, normalize=False)

144,000 samples，48000 Hz，mono float32


In [3]:
# 3. 編譯並呼叫 Project2 / SPSC / AnalysisWorker；查看各部件結果
output, stages = run_project2(recording, fs)
show_stages(stages)

部件,實際結果
輸入錄音,"144,000 frames；前 4 點：0.0000, 0.0000, 0.0000, 0.0000"
C++ callback → DSP,1125 blocks；PassthroughProcessor 原音直通
分析 SPSC,1125 blocks 送達；drop=0
AnalysisWorker / Handler,"RMS=0.01023；前 4 點：0.0000, 0.0000, 0.0000, 0.0000"
參數 SPSC,1 個固定示範快照；drop=0
callback 套用參數,1 次，revision=1；1000 Hz / 100 Hz / 0 dB；transition 5 ms
輸出音訊,"與輸入逐點相同；前 4 點：0.0000, 0.0000, 0.0000, 0.0000"


In [4]:
# 4. 播放輸出（應與原音相同；尚未加入抑制演算法）
Audio(output, rate=fs, normalize=False)